# 03 — Preprocessing Pipeline

Inputs: `data/features.csv`  
Outputs:`data/X_train.csv`, `data/X_val.csv`, `data/y_train.csv`, `data/y_val.csv`

## 0. Imports

In [8]:
import pandas as pd
import numpy as np
import warnings
import pathlib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


warnings.filterwarnings('ignore')

DATA_PATH    = '../data/features.csv'
OUTPUT_DIR   = '../data'
RANDOM_STATE = 42
TEST_SIZE    = 0.2

pathlib.Path(OUTPUT_DIR).mkdir(exist_ok=True)

## 1. Load data

In [9]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["TARGET"].value_counts(normalize=True).round(4)}')

Shape: (50000, 29)
Target distribution:
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


## 2. Define features and target

In [10]:
TARGET = 'TARGET'

binary_features = [
    'HAS_BUREAU_FILE'
]

continuous_numeric_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'TERM_MONTHS',
    'INTEREST_RATE',
    'AGE_YEARS',
    'EMPLOY_YEARS',
    'CREDIT_TO_INCOME',
    'ANNUITY_TO_INCOME',
    'DEBT_TO_INCOME',
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    'BUREAU_ACTIVE_LOANS',
    'BUREAU_CLOSED_LOANS',
    'BUREAU_TOTAL_DEBT',
    'BUREAU_TOTAL_CREDIT_LIMIT',
    'BUREAU_UTILIZATION',
    'BUREAU_DPD_30_COUNT',
    'BUREAU_DPD_60_COUNT',
    'BUREAU_DPD_90_COUNT',
    'BUREAU_INQUIRIES_6M'
]

cat_features = [
    'EMPLOYMENT_STATUS',
    'NAME_CONTRACT_TYPE',
    'HOME_OWNERSHIP',
    'LOAN_PURPOSE',
    'MONTHS_SINCE_DELINQ_BINNED'
]

all_features = binary_features + continuous_numeric_features + cat_features

X = df[all_features]
y = df[TARGET]

## 3. Stratified train/validation split

**Objective 5: Perform a stratified train/validation split**

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y        
)

print(f'Train size: {len(X_train):,} ({len(X_train)/len(X):.0%})')
print(f'Val size:   {len(X_val):,} ({len(X_val)/len(X):.0%})')
print()
print('Class distribution, train:')
print(y_train.value_counts(normalize=True).round(4))
print()
print('Class distribution, validation:')
print(y_val.value_counts(normalize=True).round(4))

Train size: 40,000 (80%)
Val size:   10,000 (20%)

Class distribution, train:
TARGET
0    0.9190
1    0.0809
Name: proportion, dtype: float64

Class distribution, validation:
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


Stratification preserves class ratio between sets.

## 4. Preprocessing Pipeline

In [12]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric',     numeric_pipeline,     continuous_numeric_features),
    ('binary',      'passthrough',      binary_features),
    ('categorical', categorical_pipeline, cat_features),
])

Continuous numeric values are median imputed and standardised. Binary values are untouched. One-hot-encode categorical features.

**Objective 6: Fit imputation and encoding on the training data**

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

print(f'X_train_processed shape: {X_train_processed.shape}')
print(f'X_val_processed shape:   {X_val_processed.shape}')

print(f'\nTrain nulls: {np.isnan(X_train_processed).sum()}')
print(f'Val nulls:   {np.isnan(X_val_processed).sum()}')

X_train_processed shape: (40000, 48)
X_val_processed shape:   (10000, 48)

Train nulls: 0
Val nulls:   0


**Objective 7. Why fitting preprocessing before splitting causes leakage**

Suppose we wish to perform median imputation for our EXT_SOURCE variables. When the imputer computes the median of `EXT_SOURCE_1` across all 50,000 rows, it uses values from what will become the validation set. Those validation statistics then flow back into the training data through the imputed values. The model is trained on data that has been subtly contaminated with information from the validation set. The same issue arises if we wanted to compute a Z-score for a variable and use the mean and standard deviation computed from the entire dataset.

## 6. Save

In [14]:
ohe_feature_names = (
    preprocessor
    .named_transformers_['categorical']
    .named_steps['encoder']
    .get_feature_names_out(cat_features)
    .tolist()
)

all_feature_names = continuous_numeric_features + binary_features + ohe_feature_names

X_train_df = pd.DataFrame(X_train_processed, columns=all_feature_names)
X_val_df   = pd.DataFrame(X_val_processed,   columns=all_feature_names)

X_train_df.to_csv('../data/X_train.csv', index=False)
X_val_df.to_csv('../data/X_val.csv',     index=False)
y_train.to_csv('../data/y_train.csv',    index=False)
y_val.to_csv('../data/y_val.csv',        index=False)

print(f'X_train: {X_train_df.shape}')
print(f'X_val:   {X_val_df.shape}')

X_train: (40000, 48)
X_val:   (10000, 48)
